# Estrutura — camada trusted (tancagem)

**Objetivo:** documentar a estrutura do parquet unificado e mapear campos nulos vs. preenchidos.

**Pré-requisito:** `py estudos/tancagem-abastecimento/pipelines/build_trusted.py`

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import (
    REPO_ROOT,
    TRUSTED_DIR,
    TRUSTED_MANIFEST,
    TRUSTED_PARQUET,
)

EXPECTED_COLS = [
    "Data",
    "NomeEmpresarial",
    "Uf",
    "Municipio",
    "Cnpj",
    "CodInstalacao",
    "Segmento",
    "DetalheInstalacao",
    "Tag",
    "TipoDaUnidade",
    "GrupoDeProdutos",
    "TancagemM3",
]
META_COLS = ["_source_file", "_source_year", "_source_period"]

if not TRUSTED_PARQUET.exists():
    raise FileNotFoundError(
        f"Parquet ausente. Execute build_trusted.py. Esperado: {TRUSTED_PARQUET}"
    )

df = pd.read_parquet(TRUSTED_PARQUET)
manifest = json.loads(TRUSTED_MANIFEST.read_text(encoding="utf-8"))

print(f"Repo: {REPO_ROOT}")
print(f"Parquet: {TRUSTED_PARQUET.relative_to(REPO_ROOT)}")
print(f"Linhas: {len(df):,} | Colunas: {len(df.columns)}")
print(f"Gerado em (manifest): {manifest.get('gerado_em')}")
print(f"Arquivos fonte: {manifest.get('arquivos')}")

## 1. Estrutura do dataset

In [ ]:
print("Colunas (ordem no parquet):")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:2}. {c} ({df[c].dtype})")

missing_expected = set(EXPECTED_COLS) - set(df.columns)
extra_cols = set(df.columns) - set(EXPECTED_COLS + META_COLS)
print(f"\nColunas esperadas ausentes: {missing_expected or 'nenhuma'}")
print(f"Colunas extras: {extra_cols or 'nenhuma'}")

In [ ]:
df.info(memory_usage="deep")

In [ ]:
display(df.head(3))
df.describe(include="all").T

## 2. Nulos e preenchimento por coluna

In [ ]:
n = len(df)

def empty_string_count(series: pd.Series) -> int:
    if series.dtype != object and not pd.api.types.is_string_dtype(series):
        return 0
    return int(series.fillna("").astype(str).str.strip().eq("").sum())

rows = []
for col in df.columns:
    null_n = int(df[col].isna().sum())
    empty_n = empty_string_count(df[col])
    filled_n = n - null_n
    rows.append(
        {
            "coluna": col,
            "dtype": str(df[col].dtype),
            "nulos": null_n,
            "%_nulos": round(100 * null_n / n, 4),
            "preenchidos": filled_n,
            "%_preenchidos": round(100 * filled_n / n, 4),
            "strings_vazias": empty_n,
            "status": "OK" if null_n == 0 and empty_n == 0 else "revisar",
        }
    )

null_report = pd.DataFrame(rows).set_index("coluna")
null_report

In [ ]:
biz_cols = EXPECTED_COLS
all_biz_null = df[biz_cols].isna().all(axis=1)
print(f"Linhas com todas as colunas de negócio nulas: {all_biz_null.sum():,}")

if all_biz_null.any():
    display(
        df.loc[all_biz_null, META_COLS]
        .value_counts("_source_file")
        .rename("linhas")
        .to_frame()
    )

In [ ]:
partial_null = df[biz_cols].isna().any(axis=1) & ~all_biz_null
print(f"Linhas com nulo parcial (alguma coluna de negócio): {partial_null.sum():,}")

if partial_null.any():
    per_col = df.loc[partial_null, biz_cols].isna().sum().sort_values(ascending=False)
    print("Nulos por coluna (somente linhas parciais):")
    display(per_col[per_col > 0])

## 3. Metadados de origem (`_source_*`)

In [ ]:
print("Snapshots distintos (_source_file):", df["_source_file"].nunique())
print("\nLinhas por arquivo fonte:")
display(df["_source_file"].value_counts().sort_index().to_frame("linhas"))

In [ ]:
print("Período Data (min → max):", df["Data"].min(), "→", df["Data"].max())
print("TancagemM3 — min/max/soma:", df["TancagemM3"].min(), df["TancagemM3"].max(), f"{df['TancagemM3'].sum():,.0f}")